**START GENERATORS**

In [0]:
import time, random
from datetime import datetime, timezone, timedelta
from concurrent.futures import ThreadPoolExecutor

# Landing paths (YOUR environment)
BASE = "/Volumes/bridge_monitoring/00_landing/landing_stream"

STREAMS = [
    (f"{BASE}/bridge_temperature", "temperature", 19.0, 23.0),
    (f"{BASE}/bridge_vibration",   "vibration",   0.005, 0.05),
    (f"{BASE}/bridge_tilt",        "tilt_angle",  -0.005, 0.005)
]

DEVICE_COUNT = 5
BATCH_INTERVAL_S = 10   # set to 60 later
LATENCY_MAX_S = 60

# Stop flag
STOP = False

def generate_stream(path, column_name, low, high, device_count, batch_interval_s, latency_max_s):
    global STOP
    while not STOP:
        now = datetime.now(timezone.utc)
        rows = []
        for device_id in range(1, device_count + 1):
            ts = now - timedelta(seconds=random.uniform(0, latency_max_s))
            val = round(random.uniform(low, high), 4)
            rows.append({"device_id": device_id, "event_time": ts, column_name: val})

        df = spark.createDataFrame(rows)
        df.write.format("delta").mode("append").save(path)
        time.sleep(batch_interval_s)

    return f"Stopped: {path}"

executor = ThreadPoolExecutor(max_workers=len(STREAMS))
futures = []

for path, col, low, high in STREAMS:
    futures.append(executor.submit(generate_stream, path, col, low, high,
                                   DEVICE_COUNT, BATCH_INTERVAL_S, LATENCY_MAX_S))

print("✅ Generators started.")
for path, _, _, _ in STREAMS:
    print(" -", path)


✅ Generators started.
 - /Volumes/bridge_monitoring/00_landing/landing_stream/bridge_temperature
 - /Volumes/bridge_monitoring/00_landing/landing_stream/bridge_vibration
 - /Volumes/bridge_monitoring/00_landing/landing_stream/bridge_tilt


**STOP GENERATORS**

In [0]:
STOP = True
print("Stopping...")

for f in futures:
    try:
        print(f.result(timeout=120))
    except Exception as e:
        print("Thread exit:", e)

executor.shutdown(wait=False)
print("✅ All generators stopped.")


Stopping...
Stopped: /Volumes/bridge_monitoring/00_landing/landing_stream/bridge_temperature
Stopped: /Volumes/bridge_monitoring/00_landing/landing_stream/bridge_vibration
Stopped: /Volumes/bridge_monitoring/00_landing/landing_stream/bridge_tilt
✅ All generators stopped.


**VALIDATE LANDING**

In [0]:
from pyspark.sql import functions as F

temp_path = f"{BASE}/bridge_temperature"
vib_path  = f"{BASE}/bridge_vibration"
tilt_path = f"{BASE}/bridge_tilt"

print("Files in temperature landing:")
display(dbutils.fs.ls(temp_path))

print("Latest temperature rows:")
display(spark.read.format("delta").load(temp_path).orderBy(F.col("event_time").desc()).limit(10))

print("Latest vibration rows:")
display(spark.read.format("delta").load(vib_path).orderBy(F.col("event_time").desc()).limit(10))

print("Latest tilt rows:")
display(spark.read.format("delta").load(tilt_path).orderBy(F.col("event_time").desc()).limit(10))


Files in temperature landing:


path,name,size,modificationTime
dbfs:/Volumes/bridge_monitoring/00_landing/landing_stream/bridge_temperature/_delta_log/,_delta_log/,0,1767280682000
dbfs:/Volumes/bridge_monitoring/00_landing/landing_stream/bridge_temperature/part-00000-0023a9f7-4f35-47b6-8ac9-874170351ce6.c000.snappy.parquet,part-00000-0023a9f7-4f35-47b6-8ac9-874170351ce6.c000.snappy.parquet,1199,1767484950000
dbfs:/Volumes/bridge_monitoring/00_landing/landing_stream/bridge_temperature/part-00000-005a2cc3-8f0f-4aa8-8b4d-1d1b185fec2d.c000.snappy.parquet,part-00000-005a2cc3-8f0f-4aa8-8b4d-1d1b185fec2d.c000.snappy.parquet,1200,1767284214000
dbfs:/Volumes/bridge_monitoring/00_landing/landing_stream/bridge_temperature/part-00000-0098fcbb-35df-4d3c-bf3e-add686538abe.c000.snappy.parquet,part-00000-0098fcbb-35df-4d3c-bf3e-add686538abe.c000.snappy.parquet,1199,1767481957000
dbfs:/Volumes/bridge_monitoring/00_landing/landing_stream/bridge_temperature/part-00000-02413912-5061-4294-829f-73ad66293eef.c000.snappy.parquet,part-00000-02413912-5061-4294-829f-73ad66293eef.c000.snappy.parquet,1200,1767485386000
dbfs:/Volumes/bridge_monitoring/00_landing/landing_stream/bridge_temperature/part-00000-02bbc7d7-41a5-4a86-8bee-f07e9fb325e4.c000.snappy.parquet,part-00000-02bbc7d7-41a5-4a86-8bee-f07e9fb325e4.c000.snappy.parquet,1199,1767485045000
dbfs:/Volumes/bridge_monitoring/00_landing/landing_stream/bridge_temperature/part-00000-0306a5a0-95aa-462c-bd08-2d8721a24b42.c000.snappy.parquet,part-00000-0306a5a0-95aa-462c-bd08-2d8721a24b42.c000.snappy.parquet,1198,1767284257000
dbfs:/Volumes/bridge_monitoring/00_landing/landing_stream/bridge_temperature/part-00000-03c0b6f8-ca4c-4834-a03d-a3ec45168728.c000.snappy.parquet,part-00000-03c0b6f8-ca4c-4834-a03d-a3ec45168728.c000.snappy.parquet,1200,1767484923000
dbfs:/Volumes/bridge_monitoring/00_landing/landing_stream/bridge_temperature/part-00000-03f06401-24d0-424d-b8f7-dc090886907e.c000.snappy.parquet,part-00000-03f06401-24d0-424d-b8f7-dc090886907e.c000.snappy.parquet,1200,1767485881000
dbfs:/Volumes/bridge_monitoring/00_landing/landing_stream/bridge_temperature/part-00000-043d448b-386f-48bc-81e4-ae397712c36a.c000.snappy.parquet,part-00000-043d448b-386f-48bc-81e4-ae397712c36a.c000.snappy.parquet,1199,1767485278000


Latest temperature rows:


device_id,event_time,temperature
1,2026-06-22T21:43:04.414Z,20.0888
2,2026-06-22T21:42:50.717Z,21.5847
4,2026-06-22T21:42:34.506Z,22.0696
5,2026-06-22T21:42:27.412Z,22.1393
5,2026-06-22T21:42:27.238Z,20.2249
3,2026-06-22T21:42:18.018Z,19.8752
1,2026-06-22T21:42:12.521Z,19.1865
3,2026-06-22T21:42:12.424Z,21.3487
4,2026-06-22T21:42:11.785Z,20.9494
2,2026-06-22T21:42:07.576Z,21.17


Latest vibration rows:


device_id,event_time,vibration
2,2026-06-22T21:43:02.672Z,0.0078
3,2026-06-22T21:42:52.398Z,0.0183
5,2026-06-22T21:42:47.550Z,0.0273
4,2026-06-22T21:42:45.879Z,0.0264
1,2026-06-22T21:42:37.771Z,0.0219
2,2026-06-22T21:42:37.187Z,0.0104
3,2026-06-22T21:42:32.819Z,0.0314
5,2026-06-22T21:42:22.832Z,0.0231
1,2026-06-22T21:42:18.785Z,0.0134
4,2026-06-22T21:42:11.852Z,0.0114


Latest tilt rows:


device_id,event_time,tilt_angle
5,2026-06-22T21:42:53.260Z,5.0E-4
1,2026-06-22T21:42:42.462Z,0.0044
4,2026-06-22T21:42:36.290Z,0.0042
3,2026-06-22T21:42:29.580Z,0.0043
3,2026-06-22T21:42:29.310Z,0.0027
4,2026-06-22T21:42:28.026Z,3.0E-4
2,2026-06-22T21:42:23.551Z,0.005
2,2026-06-22T21:42:16.360Z,-0.0024
5,2026-06-22T21:42:15.779Z,0.0031
1,2026-06-22T21:42:11.434Z,-0.0039
